# 4. Regression Models

**Pipeline position:** runs after `1_data_prep.ipynb`. Reads `panel.parquet`
and the `Master.csv` directly (to rebuild the larger sample definitions for
robustness).

## What this notebook does

1. **Headline regression (Model 3b)** — fixed-effects OLS with country-clustered
   standard errors. Regressors: human capital, capital formation, political
   stability, rule of law, production value, forestry rents, trade, four
   interaction terms (HCI × Production, GFCF × Production, HCI × Forestry,
   GFCF × Forestry), and lagged ECI. Used by chart 12.

2. **Same model across four sample definitions** — original 38-country sample,
   adjusted ≥3%, ≥2%, and ≥1%. Saves coefficients and R² for each. Used by
   charts 18 and 19.

3. **HCI × Production scatter** — pre-computes the production quartile for
   each observation so the descriptive interaction chart (13) has its data
   ready without joining.

The Lagged-ECI regressor is dropped from the headline forest plot in chart 12
because its near-unit coefficient compresses the y-axis. It is kept in the
fit so its variance is correctly partialled out.

## Outputs

- `reg_coef_main.csv` — Model 3b coefficients with 95% CI (cluster SE),
  one row per regressor, dropping the constant
- `reg_coef_across_samples.csv` — coefficients for the five key regressors
  across four sample definitions
- `reg_r2_across_samples.csv` — R² of the headline specification on each
  sample
- `hci_production_scatter.csv` — observation-level (Country, Year, log_HCI,
  ECI, production quartile)


In [1]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import statsmodels.api as sm
from types import SimpleNamespace

from _config import (ARTIFACTS, INTERMEDIARY, GULF_STATES, ECI_COL,
                     LOG_COLS, REG_VARS, INTERACT, NAME_MAP, BASE_FEATS,
                     INTERACTION_FEATS, ALL_FEATS,
                     artifact_path)

panel = pd.read_parquet(artifact_path('panel.parquet'))
print(f'Panel: {panel["Country Code"].nunique()} countries, {len(panel):,} obs')


Panel: 107 countries, 2,675 obs


## Build the regression dataset

The panel from notebook 1 already has log transforms and interaction terms.
Add lagged ECI and drop rows that can't enter the model.


In [2]:
reg_df = panel.copy()
reg_df['ECI_lag1'] = reg_df.groupby('Country Code')[ECI_COL].shift(1)


def fit_clustered(df, regressors, depvar=ECI_COL):
    cols_needed = regressors + [depvar, 'Country Code']
    sub = df[cols_needed].dropna()
    if len(sub) < 30:
        return None
    y = sub[depvar]
    X = sm.add_constant(sub[regressors])
    raw  = sm.OLS(y, X).fit()
    rob  = raw.get_robustcov_results(cov_type='cluster', groups=sub['Country Code'])
    ns = SimpleNamespace()
    ns.params   = pd.Series(rob.params, index=X.columns)
    ns.bse      = pd.Series(rob.bse,    index=X.columns)
    ns.rsquared = rob.rsquared
    ns.nobs     = rob.nobs
    return ns


## Headline model 3b

Variables in `REG_VARS` and `INTERACT`, plus lagged ECI. Save coefficients
and 95% CIs from cluster-robust standard errors.


In [3]:
plot_vars = REG_VARS + INTERACT
m3b = fit_clustered(reg_df, plot_vars + ['ECI_lag1'])
print(f'Model 3b: R² = {m3b.rsquared:.4f}, N = {int(m3b.nobs)}')

rows = []
for v in plot_vars:
    c  = float(m3b.params.get(v, np.nan))
    se = float(m3b.bse.get(v, 0))
    rows.append(dict(
        feature=v,
        short=NAME_MAP.get(v, v),
        coef=c,
        ci_lo=c - 1.96 * se,
        ci_hi=c + 1.96 * se,
        se=se,
    ))
coef_main = pd.DataFrame(rows)
coef_main.to_csv(artifact_path('reg_coef_main.csv'), index=False)
print(f'  Saved reg_coef_main.csv ({len(coef_main)} regressors)')


Model 3b: R² = 0.9140, N = 2538
  Saved reg_coef_main.csv (11 regressors)


## Coefficients across four sample definitions

The four samples are defined on the 1995 cross-section: main (the
forest-adjusted ≥1% + Gulf), and adjusted ≥3%, ≥2%, ≥1%. Re-fit Model 3a
(no lag) on each, save the five key regressors plus R².

Note: this notebook reads `Master.csv` directly to rebuild the larger
sample definitions. Notebook 1 only saves the headline sample.


In [4]:
master_full = pd.read_csv(os.path.join(INTERMEDIARY, 'Master.csv'))
master_adj = master_full.copy()
master_adj['NR_adj'] = (
    master_adj['Total natural resources rents (% of GDP)'].fillna(0)
    - master_adj['Forestry rents (% of GDP)'].fillna(0)
).clip(lower=0)


def get_sample(threshold):
    base = master_adj[master_adj['Year'] == 1995]
    selected = base.loc[base['NR_adj'] >= threshold, 'Country Code'].unique().tolist()
    for g in GULF_STATES:
        if g not in selected:
            selected.append(g)
    return selected


sample_codes = pd.read_csv(artifact_path('sample.csv'))['Country Code'].tolist()
samples = {
    'Main sample': sample_codes,
    'Adj ≥3%': get_sample(3),
    'Adj ≥2%': get_sample(2),
    'Adj ≥1%': get_sample(1),
}
for sn, sl in samples.items():
    print(f'  {sn}: {len(sl)} countries')


  Main sample: 107 countries
  Adj ≥3%: 39 countries
  Adj ≥2%: 51 countries
  Adj ≥1%: 60 countries


In [5]:
def prepare_sample_panel(slist):
    """Replicate the derived columns notebook 1 builds, but for an arbitrary sample."""
    sub = master_full[
        master_full['Year'].between(1995, 2019)
        & master_full['Country Code'].isin(slist)
    ].copy()
    sub = sub.sort_values(['Country Code', 'Year']).reset_index(drop=True)
    sub['Total_Production_Value_Per_Capita'] = (
        sub['Total_Production_Value'] / sub['Population'].replace(0, np.nan)
    )
    for c in LOG_COLS:
        if c in sub.columns:
            sub[c] = np.log1p(sub[c]).replace([np.inf, -np.inf], np.nan)
    sub['log_HCI']              = sub['Human capital index']
    sub['log_GFCF']             = sub['Gross fixed capital formation, all, Constant prices, Percent of GDP']
    sub['log_Production_Value'] = sub['Total_Production_Value_Per_Capita']
    h = sub['log_HCI'].mean()
    g = sub['log_GFCF'].mean()
    p = sub['log_Production_Value'].mean()
    f = sub['Forestry rents (% of GDP)'].mean()
    sub['log_HCI_x_log_Production']  = (sub['log_HCI']  - h) * (sub['log_Production_Value'] - p)
    sub['log_GFCF_x_log_Production'] = (sub['log_GFCF'] - g) * (sub['log_Production_Value'] - p)
    sub['log_HCI_x_forestry_rents']  = (sub['log_HCI']  - h) * (sub['Forestry rents (% of GDP)'] - f)
    sub['log_GFCF_x_forestry_rents'] = (sub['log_GFCF'] - g) * (sub['Forestry rents (% of GDP)'] - f)
    return sub


key_vars = ['log_HCI', 'log_GFCF', 'log_Production_Value',
            'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']

across_rows = []
r2_rows = []
for sn, sl in samples.items():
    sub = prepare_sample_panel(sl)
    fit = fit_clustered(sub, REG_VARS + INTERACT)
    if fit is None:
        continue
    for v in key_vars:
        c  = float(fit.params.get(v, np.nan))
        se = float(fit.bse.get(v, 0))
        across_rows.append(dict(
            sample=sn, feature=v,
            short=NAME_MAP.get(v, v),
            coef=c,
            ci_lo=c - 1.96 * se,
            ci_hi=c + 1.96 * se,
        ))
    r2_rows.append(dict(sample=sn, r2=float(fit.rsquared), n=int(fit.nobs)))

pd.DataFrame(across_rows).to_csv(artifact_path('reg_coef_across_samples.csv'), index=False)
pd.DataFrame(r2_rows).to_csv(artifact_path('reg_r2_across_samples.csv'), index=False)
print(f'  Saved reg_coef_across_samples.csv ({len(across_rows)} rows)')
print(f'  Saved reg_r2_across_samples.csv ({len(r2_rows)} rows)')


  Saved reg_coef_across_samples.csv (20 rows)
  Saved reg_r2_across_samples.csv (4 rows)


## HCI × Production scatter (chart 13)

Per-observation table with log_HCI, ECI, and the production-value quartile.
The viz notebook fits the per-quartile trendline using `numpy.polyfit`.


In [6]:
sc = panel[['Country Code', 'Country Name', 'Year', 'log_HCI',
            ECI_COL, 'Total_Production_Value_Per_Capita']].dropna().copy()
sc['Prod_Q'] = pd.qcut(sc['Total_Production_Value_Per_Capita'], 4,
                       labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])
sc.to_csv(artifact_path('hci_production_scatter.csv'), index=False)
print(f'  Saved hci_production_scatter.csv ({len(sc):,} rows)')


  Saved hci_production_scatter.csv (2,641 rows)


## Summary

| Artifact | Rows | Used by |
|---|---|---|
| reg_coef_main.csv | 11 (Model 3b regressors, no constant, no lagged ECI) | chart 12 |
| reg_coef_across_samples.csv | 5 regressors × 4 samples | chart 18 |
| reg_r2_across_samples.csv | 4 samples | chart 19 |
| hci_production_scatter.csv | observation level | chart 13 |
